In [1]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import json
import pandas as pd

In [2]:
!pip install kobert-transformers

In [4]:
base_model = BertModel.from_pretrained('monologg/kobert')

class KoBERTClassifier(nn.Module):
    def __init__(self, base_model, num_labels=2):
        super().__init__()
        self.bert = base_model
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        pooled_output = outputs.last_hidden_state[:,0,:]
        logits = self.classifier(pooled_output)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
        return loss, logits

model = KoBERTClassifier(base_model)
model.load_state_dict(torch.load('kobert_model_ver6.pt', map_location='cuda' if torch.cuda.is_available() else 'cpu'))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


KoBERTClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(8002, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [5]:
id2label = {0: "negative", 1: "positive"}

from kobert_transformers import get_tokenizer

tokenizer = get_tokenizer()

def predict_sentiment(text):
    model.eval()
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        _, logits = model(input_ids, attention_mask)
        pred = torch.argmax(logits, dim=1).item()
        return id2label[pred]

#print(predict_sentiment("민원"))


tokenizer_config.json:   0%|          | 0.00/263 [00:00<?, ?B/s]

tokenizer_78b3253a26.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

negative


In [14]:
import torch.nn.functional as F


def predict_sentiment_scores(text):
    model.eval()
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        _, logits = model(input_ids, attention_mask)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]

    # 긍정/부정 라벨 수동 지정 (0 = 부정, 1 = 긍정 순서라고 가정)
    labels = ['부정', '긍정']

    return dict(zip(labels, probs))



In [15]:
predict_sentiment_scores(["민원"])

{'부정': np.float32(0.9447238), '긍정': np.float32(0.05527615)}

In [37]:
#사용시
reviews = pd.read_csv('/content/reviews_merged_all.csv')

In [ ]:
reviews['sentiment'] = reviews['content'].apply(predict_sentiment_scores)
reviews[['부정', '긍정']] = reviews['sentiment'].apply(pd.Series)
reviews = reviews.drop(columns='sentiment')
reviews